In [2]:
import torch
x = torch.tensor([1.0, 2.0, 3.0])

print(x.shape)   
print(x.dtype)   
print(x.device)  

torch.Size([3])
torch.float32
cpu


In [3]:
x = torch.tensor(2.0, requires_grad=True)

y = x**2

y.backward()

print(x.grad)

tensor(4.)


# 📘 Module 14.1 — PyTorch Fundamentals — Notes

> **Core idea:** PyTorch uses **tensors + Autograd + computational graphs** to automatically calculate gradients, which makes neural-network training practical.

---

## 1. 🔥 Tensors

A **tensor** is PyTorch's main data structure for numerical computation.

```python
import torch

x = torch.tensor([1.0, 2.0, 3.0])
```

A tensor has important properties:

```python
x.shape
x.dtype
x.device
```

Example:

```python
x = torch.tensor([1.0, 2.0, 3.0])

print(x.shape)    # torch.Size([3])
print(x.dtype)    # torch.float32
print(x.device)   # cpu
```

### In Deep Learning

Almost everything is represented as tensors:

```text
Input Data
    ↓
Tensor
    ↓
Neural Network
    ↓
Tensor Operations
    ↓
Prediction
    ↓
Loss
    ↓
Gradients
    ↓
Updated Parameters
```

---

# 2. 🔥🔥🔥 Autograd

**Autograd = Automatic Differentiation**

It automatically calculates derivatives/gradients needed for backpropagation.

For example:

$$
y=x^2
$$

Mathematically:

$$
\frac{dy}{dx}=2x
$$

PyTorch:

```python
import torch

x = torch.tensor(2.0, requires_grad=True)

y = x**2

y.backward()

print(x.grad)
```

Output:

```text
tensor(4.)
```

Because:

$$
\frac{dy}{dx}=2(2)=4
$$

### Key idea

Instead of manually calculating:

$$
\frac{\partial L}{\partial w}
$$

for every neural-network parameter, PyTorch's **Autograd calculates it automatically**.

---

# 3. 🔥🔥🔥 Computational Graph

A **computational graph** represents the sequence of mathematical operations used to produce an output.

Example:

```python
x = torch.tensor(2.0, requires_grad=True)

z = x * 3
y = z + 1
```

Graph:

```text
       x
       │
       ▼
     × 3
       │
       ▼
       z
       │
       ▼
     + 1
       │
       ▼
       y
```

Mathematically:

$$
z=3x
$$

$$
y=z+1
$$

PyTorch records these operations so that when we call:

```python
y.backward()
```

it can move backward through the graph and apply the **chain rule**.

---

## Computational Graph + Backpropagation

Suppose:

$$
x\rightarrow z\rightarrow y\rightarrow L
$$

Then:

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\frac{\partial y}{\partial z}
\frac{\partial z}{\partial x}
$$

This is exactly the mathematical foundation of **backpropagation**.

### Important connection

```text
Chain Rule
    ↓
Computational Graph
    ↓
Backpropagation
    ↓
Gradient Calculation
    ↓
Autograd
```

🔥 **Autograd is PyTorch's automatic differentiation mechanism for performing the gradient calculations required during backpropagation.**

---

# 4. 🔥🔥🔥 `requires_grad`

When we write:

```python
x = torch.tensor(2.0, requires_grad=True)
```

we are telling PyTorch:

> **Track operations involving this tensor because I may need gradients with respect to it.**

This allows Autograd to construct/track the necessary computation history.

### With gradient tracking

```python
x = torch.tensor(2.0, requires_grad=True)

y = x**2

y.backward()

print(x.grad)
```

Result:

```text
tensor(4.)
```

### Without gradient tracking

```python
x = torch.tensor(2.0)

y = x**2
```

There is no gradient tracking for `x`.

---

# 5. 🔥 `backward()`

```python
loss.backward()
```

means, conceptually:

> **Calculate the gradients of this loss with respect to the tracked tensors that contributed to it.**

Example:

```python
x = torch.tensor(3.0, requires_grad=True)

y = x**2

y.backward()
```

Then:

```python
x.grad
```

contains:

$$
\frac{dy}{dx}=6
$$

because:

$$
\frac{d(x^2)}{dx}=2x
$$

and:

$$
2(3)=6
$$

---

# 6. 🔥 `.grad`

`.grad` stores the gradient calculated during backpropagation.

Example:

```python
w = torch.tensor(2.0, requires_grad=True)

y = w**2

y.backward()

print(w.grad)
```

Here:

```text
w
```

is the parameter/value itself.

While:

```text
w.grad
```

is:

$$
\frac{\partial y}{\partial w}
$$

### Don't confuse them

| PyTorch object    | Meaning                              |
| ----------------- | ------------------------------------ |
| `w`               | Parameter/value                      |
| `w.requires_grad` | Whether gradient tracking is enabled |
| `w.grad`          | Gradient calculated for `w`          |
| `backward()`      | Calculates gradients                 |

---

# 7. 🔥 Neural Network Example

Consider:

$$
z=wx+b
$$

and:

$$
L=(z-y)^2
$$

PyTorch:

```python
x = torch.tensor(2.0)

w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

y_true = torch.tensor(10.0)

z = w*x + b

loss = (z-y_true)**2

loss.backward()
```

The computational graph is approximately:

```text
       w ─────┐
              │
       x ─────┼──→ wx ───┐
              │           │
       b ───────────────→ + 
                           │
                           ▼
                           z
                           │
                           ▼
                         Loss
                           │
                           ▼
                      backward()
                       ↙       ↘
                   w.grad     b.grad
```

PyTorch automatically calculates:

$$
\frac{\partial L}{\partial w}
$$

and

$$
\frac{\partial L}{\partial b}
$$

---

# 8. 🔥 Parameters vs Gradients

This distinction is extremely important.

Suppose:

```python
w = torch.tensor(3.0, requires_grad=True)
```

After:

```python
loss.backward()
```

we get:

```python
w.grad
```

But **the weight hasn't been updated yet**.

The gradient calculation and parameter update are separate operations.

```text
Forward Pass
     ↓
Loss
     ↓
backward()
     ↓
Gradient
     ↓
optimizer.step()
     ↓
Updated Parameter
```

So:

> **Autograd calculates the gradient.**

> **The optimizer uses the gradient to update the parameter.**

---

# 9. 🔥 `requires_grad` in Neural Networks

In a neural network, learnable parameters such as:

* weights
* biases

need gradients during training.

Conceptually:

```text
Weights
   ↓
requires_grad=True
   ↓
Forward computation
   ↓
Loss
   ↓
Backward
   ↓
Weight gradients
   ↓
Optimizer
   ↓
Updated weights
```

Input data normally does **not** need gradient tracking during ordinary training.

```text
Weights → gradients needed ✅
Biases  → gradients needed ✅

Input   → usually no gradient needed
Labels  → no gradient needed
```

There are special applications where gradients with respect to inputs are useful, but that's not the normal training requirement.

---

# 10. 🔥 Complete PyTorch Training Mental Model

Connect everything you've learned so far:

```text
                 PYTORCH TRAINING

Input Tensor
     │
     ▼
Neural Network
     │
     ▼
Forward Pass
     │
     ▼
Prediction
     │
     ▼
Loss
     │
     ▼
loss.backward()
     │
     ▼
Autograd
     │
     ▼
Computational Graph
     │
     ▼
Gradients
     │
     ▼
parameter.grad
     │
     ▼
Optimizer
     │
     ▼
Updated Parameters
```

This is the PyTorch implementation of the mathematical training process you already learned.

---

# 11. 🧠 The Most Important Connections

### Mathematical Deep Learning

$$
\text{Forward}
\rightarrow
\text{Loss}
\rightarrow
\text{Backpropagation}
\rightarrow
\text{Gradients}
\rightarrow
\text{Parameter Update}
$$

### PyTorch

```text
model(x)
   ↓
loss
   ↓
loss.backward()
   ↓
parameter.grad
   ↓
optimizer.step()
```

### Under the hood

```text
Tensor
   ↓
requires_grad
   ↓
Operations
   ↓
Computational Graph
   ↓
Autograd
   ↓
Gradients
```

---

# ⚡ Quick Revision Table

| Concept                  | Meaning                                      |
| ------------------------ | -------------------------------------------- |
| **Tensor**               | PyTorch numerical data structure             |
| **Autograd**             | Automatic differentiation system             |
| **Computational graph**  | Records mathematical operations/dependencies |
| **`requires_grad=True`** | Track operations for gradient computation    |
| **`backward()`**         | Compute gradients backward through graph     |
| **`.grad`**              | Stores calculated gradient                   |
| **Optimizer**            | Uses gradients to update parameters          |

---

# 🎯 Interview-Level Points

### What is Autograd?

> PyTorch's automatic differentiation system that tracks operations and computes gradients needed for backpropagation.

### What does `requires_grad=True` do?

> It tells PyTorch to track operations involving the tensor so gradients can be calculated with respect to it.

### What does `loss.backward()` do?

> It performs reverse-mode automatic differentiation through the computation graph and computes gradients.

### Where are gradients stored?

> In the `.grad` attribute of tensors/parameters that require gradients.

### Does `backward()` update weights?

**No. ❌**

It calculates gradients.

```text
backward() → gradients
step()     → parameter update
```

### Is Autograd the same as the optimizer?

**No. ❌**

```text
Autograd  → calculates gradients
Optimizer → uses gradients to update parameters
```

---

# 🧠 Final Cheat Sheet

```text
TENSOR
↓
Numerical data

requires_grad=True
↓
"Track my operations"

COMPUTATIONAL GRAPH
↓
Records how operations connect

backward()
↓
Run differentiation backward

AUTOGRAD
↓
Automatically calculates gradients

.grad
↓
Stores gradients

optimizer.step()
↓
Updates parameters
```

### 🔥 One sentence to remember

> **PyTorch Autograd builds/tracks a computational graph during the forward computation and uses the chain rule during `backward()` to automatically calculate gradients, which the optimizer then uses to update model parameters.**

---
